## TESTES SIORG

In [1]:
import os
import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

bucket = os.environ["S3_BUCKET_NAME"]

s3 = boto3.client(
    "s3",
    region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
)

rows = []

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        rows.append({
            "key": obj["Key"],
            "size_bytes": obj["Size"],
            "size_mb": round(obj["Size"] / 1024**2, 2),
            "last_modified": obj["LastModified"],
        })

inventory = pd.DataFrame(rows)

inventory.sort_values("key")

,key,size_bytes,size_mb,last_modified
0,connection_test.txt,103,0.00,2026-07-08 22:29:28+00:00
1,distribuicao/distribuicao-orgaos-siorg-2022-08...,37817938,36.07,2026-07-08 22:55:29+00:00
2,distribuicao/distribuicao-orgaos-siorg-2022-09...,37822278,36.07,2026-07-08 22:55:38+00:00
3,distribuicao/distribuicao-orgaos-siorg-2022-10...,37243700,35.52,2026-07-08 22:55:45+00:00
4,distribuicao/distribuicao-orgaos-siorg-2022-11...,37019763,35.30,2026-07-08 22:55:50+00:00
...,...,...,...,...
94,iceberg/siorg/teste_conexao-bc855a0621844a068a...,3349,0.00,2026-07-13 18:20:49+00:00
95,iceberg/siorg/teste_conexao-bc855a0621844a068a...,550,0.00,2026-07-13 18:20:48+00:00
96,iceberg/siorg/teste_conexao-bc855a0621844a068a...,7081,0.01,2026-07-13 18:20:47+00:00
97,iceberg/siorg/teste_conexao-bc855a0621844a068a...,4492,0.00,2026-07-13 18:20:48+00:00


In [2]:
csv_files = inventory[
    inventory["key"].str.lower().str.endswith(".csv")
].copy()

csv_files

,key,size_bytes,size_mb,last_modified
1,distribuicao/distribuicao-orgaos-siorg-2022-08...,37817938,36.07,2026-07-08 22:55:29+00:00
2,distribuicao/distribuicao-orgaos-siorg-2022-09...,37822278,36.07,2026-07-08 22:55:38+00:00
3,distribuicao/distribuicao-orgaos-siorg-2022-10...,37243700,35.52,2026-07-08 22:55:45+00:00
4,distribuicao/distribuicao-orgaos-siorg-2022-11...,37019763,35.30,2026-07-08 22:55:50+00:00
5,distribuicao/distribuicao-orgaos-siorg-2022-12...,37125186,35.41,2026-07-08 22:55:53+00:00
...,...,...,...,...
87,estrutura-organizacional-completa/estrutura-or...,150697279,143.72,2026-07-08 22:54:22+00:00
88,estrutura-organizacional-completa/estrutura-or...,151472277,144.46,2026-07-08 22:54:31+00:00
89,estrutura-organizacional-completa/estrutura-or...,153268351,146.17,2026-07-08 22:54:42+00:00
90,estrutura-organizacional-completa/estrutura-or...,154224777,147.08,2026-07-08 22:54:54+00:00


In [3]:
csv_files["prefixo"] = csv_files["key"].str.split("/").str[0]

resumo_prefixos = (
    csv_files
    .groupby("prefixo")
    .agg(
        arquivos=("key", "count"),
        tamanho_total_gb=("size_bytes", lambda x: round(x.sum() / 1024**3, 2)),
        tamanho_medio_mb=("size_mb", "mean"),
        menor_arquivo_mb=("size_mb", "min"),
        maior_arquivo_mb=("size_mb", "max"),
    )
    .reset_index()
)

resumo_prefixos

,prefixo,arquivos,tamanho_total_gb,tamanho_medio_mb,menor_arquivo_mb,maior_arquivo_mb
0,distribuicao,45,1.66,37.764444,35.30,40.69
1,estrutura-organizacional-completa,46,5.90,131.328043,112.29,147.52


In [4]:
pd.set_option("display.max_colwidth", None)

for prefixo in csv_files["prefixo"].unique():
    print(f"\n### {prefixo}")
    display(
        csv_files[csv_files["prefixo"] == prefixo][
            ["key", "size_mb"]
        ].sort_values("key")
    )


### distribuicao


,key,size_mb
1,distribuicao/distribuicao-orgaos-siorg-2022-08.csv,36.07
2,distribuicao/distribuicao-orgaos-siorg-2022-09.csv,36.07
3,distribuicao/distribuicao-orgaos-siorg-2022-10.csv,35.52
4,distribuicao/distribuicao-orgaos-siorg-2022-11.csv,35.30
5,distribuicao/distribuicao-orgaos-siorg-2022-12.csv,35.41
6,distribuicao/distribuicao-orgaos-siorg-2023-01.csv,35.42
7,distribuicao/distribuicao-orgaos-siorg-2023-02.csv,35.30
8,distribuicao/distribuicao-orgaos-siorg-2023-03.csv,35.30
9,distribuicao/distribuicao-orgaos-siorg-2023-04.csv,35.30
10,distribuicao/distribuicao-orgaos-siorg-2023-05.csv,35.33



### estrutura-organizacional-completa


,key,size_mb
46,estrutura-organizacional-completa/estrutura-organizacional-completa-2022-08.csv,112.29
47,estrutura-organizacional-completa/estrutura-organizacional-completa-2022-09.csv,112.39
48,estrutura-organizacional-completa/estrutura-organizacional-completa-2022-10.csv,113.64
49,estrutura-organizacional-completa/estrutura-organizacional-completa-2022-11.csv,123.40
50,estrutura-organizacional-completa/estrutura-organizacional-completa-2022-12.csv,123.40
51,estrutura-organizacional-completa/estrutura-organizacional-completa-2023-01.csv,123.40
52,estrutura-organizacional-completa/estrutura-organizacional-completa-2023-02.csv,123.40
53,estrutura-organizacional-completa/estrutura-organizacional-completa-2023-03.csv,123.40
54,estrutura-organizacional-completa/estrutura-organizacional-completa-2023-04.csv,123.40
55,estrutura-organizacional-completa/estrutura-organizacional-completa-2023-05.csv,123.57


In [5]:
import io
import pandas as pd

def ler_amostra_s3(key, linhas=10):
    obj = s3.get_object(
        Bucket=bucket,
        Key=key,
    )

    raw = obj["Body"].read(5_000_000)

    for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            texto = raw.decode(encoding)

            df = pd.read_csv(
                io.StringIO(texto),
                sep=None,
                engine="python",
                nrows=linhas,
            )

            return df, encoding

        except Exception:
            continue

    raise ValueError(f"Não foi possível ler {key}")

In [6]:
amostras = {}

for prefixo in csv_files["prefixo"].unique():
    key = (
        csv_files[csv_files["prefixo"] == prefixo]
        .sort_values("key")
        .iloc[0]["key"]
    )

    df, encoding = ler_amostra_s3(key)

    amostras[prefixo] = df

    print(f"\n### {prefixo}")
    print(f"Arquivo: {key}")
    print(f"Encoding: {encoding}")
    print(f"Colunas: {list(df.columns)}")

    display(df.head())


### distribuicao
Arquivo: distribuicao/distribuicao-orgaos-siorg-2022-08.csv
Encoding: utf-8-sig
Colunas: ['nivel_hierarquico', 'cod_orgao_entidade', 'nome_orgao_entidade', 'sigla_orgao_entidade', 'natureza_juridica', 'subnatureza_juridica', 'poder', 'esfera', 'cod_unidade_pai', 'cod_unidade', 'nome_unidade', 'sigla_unidade', 'endereco', 'endereco_complemento', 'bairro', 'municipio', 'uf', 'cep', 'telefone', 'email', 'area_atuacao', 'nivel_normatizacao', 'autonomia_gestao_cargos', 'tipo_cargo', 'categoria_cargo', 'nivel_cargo', 'denominacao_cargo', 'complemento_denominacao', 'mobilidade', 'obriga_distribuicao', 'compoe_estrutura', 'autoridade', 'regra_autoridade', 'regra_cargo_nome_unidade', 'temporario', 'cod_instancia']


,nivel_hierarquico,cod_orgao_entidade,nome_orgao_entidade,sigla_orgao_entidade,natureza_juridica,subnatureza_juridica,poder,esfera,cod_unidade_pai,cod_unidade,...,denominacao_cargo,complemento_denominacao,mobilidade,obriga_distribuicao,compoe_estrutura,autoridade,regra_autoridade,regra_cargo_nome_unidade,temporario,cod_instancia
0,1,200000,República Federativa do Brasil,,NaN,NaN,NaN,NaN,NaN,200000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,208614,Distrito Federal,DF,NaN,NaN,NaN,Estadual/Distrital,200000.0,208614,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,208615,Poder Executivo,PE,NaN,NaN,Executivo,Estadual/Distrital,208614.0,208615,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,200002,Governo do Distrito Federal,GDF,Administração Direta,NaN,Executivo,Estadual/Distrital,208615.0,200002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,266982,Polícia Civil do Distrito Federal,PCDF,Administração Direta,NaN,Executivo,Estadual/Distrital,200002.0,266982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



### estrutura-organizacional-completa
Arquivo: estrutura-organizacional-completa/estrutura-organizacional-completa-2022-08.csv
Encoding: utf-8-sig
Colunas: ['codigoUnidade', 'codigoUnidadePai', 'codigoOrgaoEntidade', 'codigoTipoUnidade', 'nome', 'sigla', 'codigoEsfera', 'codigoPoder', 'codigoNaturezaJuridica', 'codigoSubNaturezaJuridica', 'nivelNormatizacao', 'versaoConsulta', 'dataInicialVersaoConsulta', 'dataFinalVersaoConsulta', 'operacao', 'codigoUnidadePaiAnterior', 'codigoOrgaoEntidadeAnterior', 'regulamentoEspecifico', 'codigoCategoriaUnidade', 'competencia', 'finalidade', 'missao', 'descricaoAtoNormativo', 'areaAtuacao', 'telefone', 'email', 'site', 'linhaEndereco', 'bairro', 'cep', 'uf', 'municipio', 'pais', 'horarioFuncionamento']


,codigoUnidade,codigoUnidadePai,codigoOrgaoEntidade,codigoTipoUnidade,nome,sigla,codigoEsfera,codigoPoder,codigoNaturezaJuridica,codigoSubNaturezaJuridica,...,telefone,email,site,linhaEndereco,bairro,cep,uf,municipio,pais,horarioFuncionamento
0,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200000,NaN,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200000,http://estruturaorganizacional.dados.gov.br/id/tipo-unidade/ente,República Federativa do Brasil,,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200001,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200000,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200001,http://estruturaorganizacional.dados.gov.br/id/tipo-unidade/ente,União,UNIAO,http://estruturaorganizacional.dados.gov.br/id/esfera/1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200002,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/208615,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/200002,http://estruturaorganizacional.dados.gov.br/id/tipo-unidade/orgao,Governo do Distrito Federal,GDF,http://estruturaorganizacional.dados.gov.br/id/esfera/2,http://estruturaorganizacional.dados.gov.br/id/poder/1,http://estruturaorganizacional.dados.gov.br/id/natureza-juridica/3,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/36554,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/1,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/1,http://estruturaorganizacional.dados.gov.br/id/tipo-unidade/unidade-colegiada,Fundo Nacional de Desenvolvimento Científico e Tecnológico,FNDCT,http://estruturaorganizacional.dados.gov.br/id/esfera/1,http://estruturaorganizacional.dados.gov.br/id/poder/1,http://estruturaorganizacional.dados.gov.br/id/natureza-juridica/1,NaN,...,(55) (21) 25550330,presidencia@finep.gov.br,www.finep.gov.br,"Av. República do Chile, 330, Torre Oeste , 10º, 11º, 12º, 15º, 16º e 17º andares",Centro,20031170.0,RJ,3304557.0,NaN,NaN
4,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/84172,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/1,http://estruturaorganizacional.dados.gov.br/id/unidade-organizacional/1,http://estruturaorganizacional.dados.gov.br/id/tipo-unidade/unidade-colegiada,Conselho Consultivo,CC-FINEP,http://estruturaorganizacional.dados.gov.br/id/esfera/1,http://estruturaorganizacional.dados.gov.br/id/poder/1,http://estruturaorganizacional.dados.gov.br/id/natureza-juridica/1,NaN,...,(55) (21) 25550330,presidencia@finep.gov.br,www.finep.gov.br,"Av. República do Chile, 330, Torre Oeste , 10º, 11º, 12º, 15º, 16º e 17º andares",Centro,20031170.0,RJ,3304557.0,NaN,NaN


In [7]:
for prefixo, df in amostras.items():
    print(f"\n### Perfil: {prefixo}")

    perfil = pd.DataFrame({
        "coluna": df.columns,
        "tipo_pandas": df.dtypes.astype(str).values,
        "nulos_amostra": df.isna().sum().values,
        "distintos_amostra": df.nunique(dropna=True).values,
    })

    display(perfil)


### Perfil: distribuicao


,coluna,tipo_pandas,nulos_amostra,distintos_amostra
0,nivel_hierarquico,int64,0,8
1,cod_orgao_entidade,int64,0,5
2,nome_orgao_entidade,object,0,5
3,sigla_orgao_entidade,object,0,5
4,natureza_juridica,object,3,1
5,subnatureza_juridica,float64,10,0
6,poder,object,2,1
7,esfera,object,1,1
8,cod_unidade_pai,float64,1,7
9,cod_unidade,int64,0,10



### Perfil: estrutura-organizacional-completa


,coluna,tipo_pandas,nulos_amostra,distintos_amostra
0,codigoUnidade,object,0,10
1,codigoUnidadePai,object,1,5
2,codigoOrgaoEntidade,object,0,4
3,codigoTipoUnidade,object,0,5
4,nome,object,0,10
5,sigla,object,0,9
6,codigoEsfera,object,1,2
7,codigoPoder,object,2,1
8,codigoNaturezaJuridica,object,2,2
9,codigoSubNaturezaJuridica,float64,10,0
